In [ ]:
# pip install git+https://github.com/lsst-sims/rubin_nights.git
import os
import numpy as np

from rubin_nights.connections import get_access_token
from rubin_nights.consdb_query import ConsDbTap, ConsDbFastAPI

on_rsp = False
# Are you on an RSP?
if on_rsp:
    api_base = os.getenv("EXTERNAL_INSTANCE_URL", "")
    token = get_access_token()
# Or are you outside of an RSP? - just use USDF and your own USDF-RSP token
# See https://rsp.lsst.io/guides/auth/creating-user-tokens.html
else:
    api_base = "https://usdf-rsp.slac.stanford.edu"
    token = get_access_token("/Users/lynnej/.lsst/usdf_rsp")
    
consdb_tap = ConsDbTap(api_base=api_base, token=token)
consdb_fastapi = ConsDbFastAPI(api_base=api_base, auth=('user', token))

It's worth noting that the queries below are much faster when run at the RSP, and also FastAPI is faster than TAP performs better. 

In [ ]:
query = "select * from cdb_lsstcam.visit1 where science_program = 'BLOCK-365'"

In [ ]:
%%timeit
visits = consdb_fastapi.query(query)

In [ ]:
%%timeit
visits = consdb_tap.query(query)

In [ ]:
%%timeit
visits = consdb_tap.tap.search(query)

In [ ]:
np.all(visits['visit_id'] == visits2['visit_id'])

In [ ]:
visits.head()[['visit_id', 'obs_start', 's_ra', 's_dec', 'sky_rotation', 'band', 'target_name', 'observation_reason']]

In [ ]:
visits.groupby(['target_name', 'band']).agg({'obs_start': ('first', 'last', 'count')})